In [ ]:
from pathlib import Path
import pandas as pd

data = pd.read_csv(
    "../../data/processed/vascular_vessels.csv"
)
node_a = data.get("node_a")
node_b = data.get("node_b")

list = []
for i, j in zip(node_a, node_b):
    if i not in list:
        list.append(i)
    if j not in list:
        list.append(j)
print()
print(len(list))

4046


In [ ]:


source = "../../data/ADAN_dataset.csv"

df = pd.read_csv(
    source,
    sep=";",
    engine="python",
    skipinitialspace=True
)

# Clean columns
df.columns = df.columns.str.strip()
df = df.loc[:, df.columns != ""]

for col in df.select_dtypes(include="str"):
    df[col] = df[col].str.strip()

# Build vessel representation
vessels = pd.DataFrame({
    "id": range(len(df)),

    "name": df["#[1] vessel name"],

    "node_a": df["[2] first node"],
    "node_b": df["[3] second node"],

    "length_cm": df["[4] length [cm]"],
    "radius_a_cm": df["[5] inlet radius [cm]"],
    "radius_b_cm": df["[6] outlet radius [cm]"],

    "type": df["[7] vessel type (1 artery, 0 vein)"].map({
        0: "vein",
        1: "artery",
    }),

    "region": df["[8] vessel region"],

    "perforator": (
        df["[9] perforator (1 yes, -1 no)"] == 1
    ),

    "vascular_territory": df["[12] vascular territory"],
    "connection_a": df["[13] first connection"],
    "connection_b": df["[14] second connection"],
})

# Normalize missing integer-like values
for col in [
    "vascular_territory",
    "connection_a",
    "connection_b",
]:
    vessels[col] = (
        pd.to_numeric(vessels[col], errors="coerce")
        .fillna(-1)
        .astype(int)
    )

# Build node representation from actual vessel connectivity
node_ids = sorted(
    set(vessels["node_a"]) |
    set(vessels["node_b"])
)

nodes = pd.DataFrame({
    "id": node_ids
})

# Validation
assert len(vessels) == 4402
assert len(nodes) == 4046

assert vessels["id"].is_unique
assert nodes["id"].is_unique

valid_nodes = set(nodes["id"])

assert vessels["node_a"].isin(valid_nodes).all()
assert vessels["node_b"].isin(valid_nodes).all()

assert (vessels["length_cm"] > 0).all()
assert (vessels["radius_a_cm"] > 0).all()
assert (vessels["radius_b_cm"] > 0).all()

assert vessels["type"].isin(["artery", "vein"]).all()

print("Dataset validation passed.")
print(f"Nodes:   {len(nodes):,}")
print(f"Vessels: {len(vessels):,}")



Dataset validation passed.
Nodes:   4,046
Vessels: 4,402


In [2]:
patterns = [
    "aorta",
    "vena_cava",
    "cava",
    "atrium",
    "ventricle",
]

mask = False

for pattern in patterns:
    current = vessels["name"].str.contains(
        pattern,
        case=False,
        na=False
    )

    mask = current if isinstance(mask, bool) else (mask | current)

candidates = vessels[mask][
    [
        "id",
        "name",
        "node_a",
        "node_b",
        "type",
        "length_cm",
        "radius_a_cm",
        "radius_b_cm",
    ]
]

candidates

,id,name,node_a,node_b,type,length_cm,radius_a_cm,radius_b_cm
799,799,A_thoracic_aorta_C_799,761,762,artery,0.989803,1.055000,1.036380
800,800,A_thoracic_aorta_C_800,762,763,artery,0.788038,1.036380,1.021555
801,801,A_thoracic_aorta_C_801,763,764,artery,1.555611,1.021555,0.992290
802,802,A_thoracic_aorta_C_802,764,765,artery,0.532705,0.992290,0.982269
803,803,A_thoracic_aorta_C_803,765,766,artery,1.856307,0.982269,0.947347
804,804,A_thoracic_aorta_C_804,766,767,artery,0.446066,0.947347,0.938956
805,805,A_thoracic_aorta_C_805,767,768,artery,1.724297,0.938956,0.906518
806,806,A_thoracic_aorta_C_806,768,769,artery,0.565208,0.906518,0.895885
807,807,A_thoracic_aorta_C_807,769,770,artery,1.813464,0.895885,0.861769
808,808,A_thoracic_aorta_C_808,770,771,artery,0.583184,0.861769,0.850798


In [3]:
largest_arteries = (
    vessels[vessels["type"] == "artery"]
    .sort_values("radius_a_cm", ascending=False)
    [
        [
            "id",
            "name",
            "node_a",
            "node_b",
            "radius_a_cm",
            "radius_b_cm",
        ]
    ]
    .head(20)
)

largest_veins = (
    vessels[vessels["type"] == "vein"]
    .sort_values("radius_a_cm", ascending=False)
    [
        [
            "id",
            "name",
            "node_a",
            "node_b",
            "radius_a_cm",
            "radius_b_cm",
        ]
    ]
    .head(20)
)

print("Largest arteries:")
display(largest_arteries)

print("Largest veins:")
display(largest_veins)

Largest arteries:


,id,name,node_a,node_b,radius_a_cm,radius_b_cm
815,815,A_aortic_arch_C_815,778,779,1.595000,1.533368
816,816,A_aortic_arch_C_816,779,780,1.533368,1.521267
817,817,A_aortic_arch_C_817,780,673,1.521267,1.295488
818,818,A_aortic_arch_C_818,673,734,1.295488,1.256847
819,819,A_aortic_arch_C_819,734,578,1.256847,1.228778
820,820,A_aortic_arch_C_820,578,781,1.228778,1.180097
821,821,A_aortic_arch_C_821,781,782,1.180097,1.156243
822,822,A_aortic_arch_C_822,782,783,1.156243,1.131303
823,823,A_aortic_arch_C_823,783,784,1.131303,1.108389
824,824,A_aortic_arch_C_824,784,785,1.108389,1.094266


Largest veins:


,id,name,node_a,node_b,radius_a_cm,radius_b_cm
4149,4149,V_inferior_vena_cava_C_4149,3803,3809,0.987281,0.995000
4148,4148,V_inferior_vena_cava_C_4148,3808,3803,0.966014,0.987281
4147,4147,V_inferior_vena_cava_C_4147,3801,3808,0.947872,0.966014
4146,4146,V_inferior_vena_cava_C_4146,3799,3801,0.946606,0.947872
4145,4145,V_inferior_vena_cava_C_4145,3807,3799,0.896000,0.946606
4143,4143,VT_hepatic_vein_C_VOG11_T1_C_4143,3803,3804,0.696500,0.696500
4286,4286,V_superior_vena_cava_C_4286,3931,3805,0.650000,0.650000
4287,4287,V_superior_vena_cava_C_4287,3805,3934,0.650000,0.650000
4150,4150,V_common_iliac_vein_R_4150,3807,3810,0.631400,0.631400
4151,4151,V_common_iliac_vein_L_4151,3807,3811,0.631400,0.631400


In [4]:
def connected_vessels(node_id):
    return vessels[
        (vessels["node_a"] == node_id) |
        (vessels["node_b"] == node_id)
    ][
        [
            "id",
            "name",
            "node_a",
            "node_b",
            "type",
            "radius_a_cm",
            "radius_b_cm"
        ]
    ]


# Investigate both ends of the proximal aortic arch
print("Node 778:")
display(connected_vessels(778))

print("Node 779:")
display(connected_vessels(779))


# Investigate apparent IVC endpoint
print("Node 3809:")
display(connected_vessels(3809))


# Investigate apparent SVC endpoint
print("Node 3934:")
display(connected_vessels(3934))

Node 778:


,id,name,node_a,node_b,type,radius_a_cm,radius_b_cm
815,815,A_aortic_arch_C_815,778,779,artery,1.595,1.533368


Node 779:


,id,name,node_a,node_b,type,radius_a_cm,radius_b_cm
815,815,A_aortic_arch_C_815,778,779,artery,1.595000,1.533368
816,816,A_aortic_arch_C_816,779,780,artery,1.533368,1.521267
1579,1579,A_left_coronary_C_1579,779,1471,artery,0.225000,0.225000


Node 3809:


,id,name,node_a,node_b,type,radius_a_cm,radius_b_cm
4149,4149,V_inferior_vena_cava_C_4149,3803,3809,vein,0.987281,0.995


Node 3934:


,id,name,node_a,node_b,type,radius_a_cm,radius_b_cm
4287,4287,V_superior_vena_cava_C_4287,3805,3934,vein,0.65,0.65


In [5]:
from collections import Counter

node_degree = Counter(vessels["node_a"])
node_degree.update(vessels["node_b"])

major = vessels[
    vessels["name"].str.contains(
        "aort|vena_cava",
        case=False,
        na=False
    )
].copy()

major["degree_a"] = major["node_a"].map(node_degree)
major["degree_b"] = major["node_b"].map(node_degree)

display(
    major[
        (major["degree_a"] == 1) |
        (major["degree_b"] == 1)
    ][
        [
            "id",
            "name",
            "node_a",
            "degree_a",
            "node_b",
            "degree_b",
            "type",
            "radius_a_cm",
            "radius_b_cm"
        ]
    ]
)

,id,name,node_a,degree_a,node_b,degree_b,type,radius_a_cm,radius_b_cm
815,815,A_aortic_arch_C_815,778,1,779,3,artery,1.595000,1.533368
4149,4149,V_inferior_vena_cava_C_4149,3803,3,3809,1,vein,0.987281,0.995000
4287,4287,V_superior_vena_cava_C_4287,3805,3,3934,1,vein,0.650000,0.650000
